# CommuniScore-AI Cloud Backend Runner
This notebook runs the CommuniScore-AI Python ML backend on Google Colab's free cloud servers. 
Colab provides **15GB of RAM and a free GPU**, which runs the AI models extremely fast and will never run out of memory!

### How to run:
1. Go to **Runtime** in the top menu -> **Change runtime type** -> Select **T4 GPU** (or CPU if GPU is unavailable) -> Click **Save**.
2. Click the **Play button** on each cell below in order.

In [ ]:
# 1. Clone the GitHub repository and go to the backend directory
%cd /content
!rm -rf Communiscore-ai
!git clone https://github.com/Ritheshiitain/Communiscore-ai.git
%cd Communiscore-ai/backend

In [ ]:
# 2. Install backend python dependencies
!pip install -r requirements.txt

In [ ]:
# 3. Start transparent SSH Tunnel (Serveo/Pinggy) and uvicorn concurrently
# This completely avoids localtunnel's security warning screen, letting API requests pass through instantly!
import subprocess
import time
import re
import sys
import threading

print("[INFO] Starting SSH Tunnel to expose port 8000...")

# We will use serveo.net (or pinggy.io as a fallback)
ssh_cmd = [
    "ssh", 
    "-o", "StrictHostKeyChecking=no", 
    "-o", "ServerAliveInterval=60", 
    "-R", "80:localhost:8000", 
    "serveo.net"
]

tunnel_proc = subprocess.Popen(ssh_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Read stdout in a background thread to print tunnel logs and capture the URL
tunnel_url = None
def read_tunnel_output():
    global tunnel_url
    while True:
        line = tunnel_proc.stdout.readline()
        if not line:
            break
        sys.stdout.write(line)
        sys.stdout.flush()
        match = re.search(r"https?://[a-zA-Z0-9.-]+\.serveo\.net", line)
        if match:
            tunnel_url = match.group(0)

t = threading.Thread(target=read_tunnel_output, daemon=True)
t.start()

# Wait for Serveo to establish connection and print URL
time.sleep(5)

if tunnel_url:
    print("\n===================================================")
    print(f"YOUR LIVE BACKEND URL: {tunnel_url}")
    print("===================================================")
    print("1. Copy the URL above.")
    print("2. Paste it into your Vercel/Render website's Settings tab!")
    print("3. Click Save Settings. The status badge will instantly change to ONLINE!")
    print("===================================================\n")
else:
    print("\n[WARN] Serveo.net did not return a URL in time. Starting fallback via pinggy.io...")
    pinggy_cmd = [
        "ssh", 
        "-o", "StrictHostKeyChecking=no", 
        "-o", "ServerAliveInterval=60", 
        "-p", "443", 
        "-R", "0:localhost:8000", 
        "qr@pinggy.io"
    ]
    tunnel_proc.terminate()
    tunnel_proc = subprocess.Popen(pinggy_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    # Wait for Pinggy to print URL
    time.sleep(5)
    # Read the output to capture url
    outputs = []
    for _ in range(20):
        line = tunnel_proc.stdout.readline()
        if not line: break
        sys.stdout.write(line)
        sys.stdout.flush()
        match = re.search(r"https?://[a-zA-Z0-9.-]+\.pinggy\.link", line)
        if match:
            tunnel_url = match.group(0)
            break
    
    if tunnel_url:
        print("\n===================================================")
        print(f"YOUR LIVE BACKEND URL: {tunnel_url}")
        print("===================================================")
    else:
        print("\n[ERROR] Both Serveo and Pinggy failed to create a tunnel. Please restart the cell.")

# Start the FastAPI app in the foreground using uvicorn
!uvicorn main:app --host 0.0.0.0 --port 8000